In [0]:
%sql
USE CATALOG shopsphere;

CREATE TABLE IF NOT EXISTS shopsphere.silver.order_items(
   order_item_id INT,
    order_id INT,
    product_id INT,
    quantity INT,
    Unit_price DOUBLE,
    discount DOUBLE,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
);

CREATE TABLE IF NOT EXISTS shopsphere.quarantine.order_items(
    order_item_id INT,
    order_id INT,
    product_id INT,
    quantity INT,
    Unit_price DOUBLE,
    discount DOUBLE,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA;

In [0]:

from pyspark.sql.functions import *
from delta.tables import *

df_orderItems = spark.read.table("shopsphere.bronze.order_items")


In [0]:
#Calculate net amount

df_orderItems = df_orderItems.withColumn("gross_amount",col("quantity") * col("unit_price")).withColumn("discount_amount",col("gross_amount") * col("discount")).withColumn("net_amount",col("gross_amount") - col("discount_amount"))


In [0]:
#validation for quantity, unitprice,discount
df_valid = df_orderItems.filter(
    (col("quantity") > 0)
    & (col("unit_price") >= 0)
    & (col("discount") >= 0)
    & (col("discount") <= 1)
)

df_quarantine = df_orderItems.filter(
    (~((col("quantity") > 0) & (col("unit_price") >= 0) & (col("discount") >= 0) & (col("discount") <= 1)))).withColumn("validation_status", lit("invalid_quantity_unitprice_discount"))


In [0]:
#quarantine rejected rows from validation
quarantine_table = DeltaTable.forName(spark,"shopsphere.quarantine.order_items")

quarantine_table.alias("target").merge(df_quarantine.alias("source"),
                                       "target.order_item_id = source.order_item_id").\
                                        whenNotMatchedInsertAll().\
                                        withSchemaEvolution().\
                                        execute()

In [0]:
#write transformed data to silver table
silver_table = DeltaTable.forName(spark,"shopsphere.silver.order_items")

silver_table.alias("target").merge(df_valid.alias("source"),
                                       "target.order_item_id = source.order_item_id").\
                                        whenNotMatchedInsertAll().\
                                        withSchemaEvolution().\
                                        execute()
